# H1/H2/H3 grid — Conformal Burden v2 (cheap-first last-layer arms), then **STOP**

Spec §3/§6. Last-layer arms (ERM, DFR, AFR, last-layer-GroupDRO, balanced subsampling) on
**Waterbirds + CelebA**, two frozen backbones (**ERM-ResNet-50** + **CLIP ViT-B/32**),
**3 seeds × 10 calibration splits**, APS/RAPS/THR, **full ρ sweep**.

Reports (spec §8): **H1 accuracy-matched** divergence (primary; raw = "uncontrolled"), **H2**
ranking inversion **with CIs** (Task B — an inversion is real only if the burden-top vs accuracy-top
cov_gap CI excludes 0), **H3 burden survival** (Task A — divergence survival + set-size-disparity
relocation, coverage stability reported separately). → `RESULTS_study.md` + CSVs + figures.

**STOPS** before the optional heavy full-GroupDRO fine-tune and any 3rd/4th dataset. Every arm
carries the §2 gates; sub-floor arms are excluded with a reason (BLOCKERS), never shipped.

## 0. Parameters — **EDIT THESE**

In [ ]:
# ===================== EDIT THESE =====================
REPO_SOURCE   = "git"
REPO_URL      = "https://github.com/octadion/vgscp.git"   # EDIT
REPO_BRANCH   = "main"
REPO_DRIVE_ZIP= "/content/drive/MyDrive/vgscp.zip"
DRIVE_CACHE   = "/content/drive/MyDrive/vgscp_cache"

SEEDS         = 3
N_SPLITS      = 10
RESNET_EPOCHS = 10
# CelebA ERM-ResNet budget (spec <5h): random subsample of the composited train split. Random
# (not class-balanced) PRESERVES the in-domain spurious correlation; the §2 gate re-verifies.
CELEBA_RESNET_MAX_TRAIN = 30000

WATERBIRDS_URL = "https://nlp.stanford.edu/data/dro/waterbird_complete95_forest2water2.tar.gz"
CELEBA_SOURCE  = "kaggle"  # "kaggle" (default; needs kaggle.json) | "drive" | "skip"
CELEBA_DRIVE   = ""        # used only if CELEBA_SOURCE=="drive": pre-extracted CelebA folder on Drive
# ======================================================
import os, sys, time, subprocess
def sh(cmd, **kw):
    print("$", cmd); return subprocess.run(cmd, shell=True, **kw)

## 1. GPU + install

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
subprocess.run("pip -q install open_clip_torch ftfy regex tqdm pyyaml scikit-learn scipy pandas matplotlib torchvision", shell=True)

## 2. Mount Drive + get repo

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
os.makedirs(DRIVE_CACHE, exist_ok=True)
REPO_DIR = "/content/vgscp"
if REPO_SOURCE == "git":
    sh(f"rm -rf {REPO_DIR} && git clone --branch {REPO_BRANCH} {REPO_URL} {REPO_DIR}")
else:
    sh(f"rm -rf {REPO_DIR} && mkdir -p {REPO_DIR} && unzip -q {REPO_DRIVE_ZIP} -d {REPO_DIR}")
os.chdir(REPO_DIR); sys.path.insert(0, REPO_DIR)
print("repo:", os.getcwd())

## 3. Datasets + env vars (Waterbirds auto; CelebA via Kaggle)
Waterbirds is wget-ed automatically. **CelebA** (`CELEBA_SOURCE="kaggle"`, default) downloads `jessicali9530/celeba-dataset`, caches the zip to Drive, and converts its CSV metadata to the `.txt` format the loader expects. **You must upload `kaggle.json` once** (Colab: Files panel -> upload to `/content`; get it at kaggle.com -> Account -> Create New API Token). Alternatives: `CELEBA_SOURCE="drive"` (set `CELEBA_DRIVE` to a pre-extracted CelebA folder) or `"skip"` (Waterbirds-only). gdrive-by-ID was deliberately NOT used (quota-unreliable).

In [ ]:
from study_robust_train.colab_data import prepare_waterbirds, prepare_celeba

os.environ["WATERBIRDS_ROOT"] = prepare_waterbirds(DRIVE_CACHE, WATERBIRDS_URL)
CELEBA_ROOT = prepare_celeba(DRIVE_CACHE, source=CELEBA_SOURCE, celeba_drive=CELEBA_DRIVE)
CELEBA_OK = bool(CELEBA_ROOT) and os.path.isdir(CELEBA_ROOT)
if CELEBA_OK:
    os.environ["CELEBA_ROOT"] = CELEBA_ROOT; print("CelebA root:", CELEBA_ROOT)
else:
    print(f"[note] CelebA not available (source={CELEBA_SOURCE}) -> SKIPPED (Waterbirds-only, logged).")

# persist feature caches to Drive (so build_griddata hits cache across sessions)
for c in ("cache_clip", "cache_resnet"):
    sh(f"rm -rf results/{c}"); os.makedirs(f"{DRIVE_CACHE}/{c}", exist_ok=True); os.makedirs("results", exist_ok=True)
    sh(f"ln -s {DRIVE_CACHE}/{c} results/{c}")
print("WATERBIRDS_ROOT=", os.environ.get("WATERBIRDS_ROOT"))

## 4. Build GridData (extract + cache features) per (dataset × backbone)
CelebA ERM-ResNet trains on a documented **random subsample** (`CELEBA_RESNET_MAX_TRAIN`) to stay
<5h — random preserves the in-domain spurious correlation; the §2 worst-group gate re-verifies.

In [ ]:
from study_robust_train.datasets import build_griddata

def cfg_for(dataset):
    base = {"clip": {"model_name": "ViT-B-32", "pretrained": "openai", "device": "cuda",
                     "cache_dir": "results/cache_clip"},
            "resnet": {"device": "cuda", "epochs": RESNET_EPOCHS, "lr": 1e-3, "batch_size": 128,
                       "cache_dir": "results/cache_resnet"}}
    if dataset == "waterbirds":
        base["dataset"] = {"root": os.environ["WATERBIRDS_ROOT"], "image_size": 224,
                           "n_classes": 2, "download": False}
    else:
        base["dataset"] = {"root": os.environ["CELEBA_ROOT"], "n_classes": 2}
        base["resnet"]["max_train"] = CELEBA_RESNET_MAX_TRAIN   # documented subsample (spec budget)
    return base

KEYS = [("waterbirds", "resnet50_erm"), ("waterbirds", "clip_vitb32")]
if CELEBA_OK:
    KEYS += [("celeba", "resnet50_erm"), ("celeba", "clip_vitb32")]

data, skipped = {}, []
for ds, bb in KEYS:
    try:
        t = time.time(); gd = build_griddata(ds, bb, cfg_for(ds), seed=0); data[(bb, ds)] = gd
        print(f"[built] {bb}/{ds}: train {gd.train[0].shape}, eval {gd.eval_domain[0].shape} "
              f"({(time.time()-t)/60:.1f} min)")
    except Exception as e:
        skipped.append((bb, ds, str(e))); print(f"[SKIP] {bb}/{ds}: {e}")
print("built:", list(data.keys()), "| skipped:", [(b,d) for b,d,_ in skipped])

## 5. Run the grid + verdicts; write RESULTS_study.md + CSV + figures

In [ ]:
from study_robust_train.grid import run_grid, write_csv, write_results_md
from study_robust_train.figures import make_figures

t = time.time()
out = run_grid(data, seeds=tuple(range(SEEDS)), n_splits=N_SPLITS)
print(f"[grid] {len(out['records'])} records, {len(out['excluded'])} excluded, "
      f"{len(out['flagged'])} flagged ({(time.time()-t)/60:.1f} min)")
os.makedirs("results/study", exist_ok=True)
write_csv(out["records"], "results/study/grid_records.csv")
write_results_md(out, "RESULTS_study.md", synthetic=False)
figs = make_figures(out, "results/study/figures")
print("wrote RESULTS_study.md, results/study/grid_records.csv,", len(figs), "figures")

## 6. §2 gate check + verdict summary (H1 GO / H2 inversion-REAL / H3 labels)

In [ ]:
import numpy as np
print("=== worst-group accuracy per arm (§2 gate) ===")
print("    CelebA refs: DFR worst-group >=0.85 ideal (hard floor 0.80, soft-flag <0.85); ERM ~0.4-0.5")
for (bb, ds) in data:
    for m in sorted({r["method"] for r in out["records"] if (r["backbone"],r["dataset"])==(bb,ds)}):
        wg = np.mean([r["worst_group_acc"] for r in out["records"]
                      if (r["backbone"],r["dataset"])==(bb,ds) and r["method"]==m])
        print(f"  {bb}/{ds:11s} {m:18s} worst-group acc = {wg:.3f}")
for label, lst in [("EXCLUDED (hard floor)", out["excluded"]), ("FLAGGED (soft, kept)", out["flagged"])]:
    if lst:
        print(f"\n=== {label} ===")
        for e in lst:
            ref = e.get("floor", e.get("expected_min"))
            print(f"  {e['backbone']}/{e['dataset']} {e['method']} seed{e['seed']}: {e['worst_group_acc']:.3f} (ref {ref})")

print("\n=== H1 / H2 / H3 ===")
for key, v in out["verdicts"].items():
    if "note" in v: print(key, "->", v["note"]); continue
    h1go = {m: mr["GO"] for m, mr in v["h1"]["methods"].items()}
    h2 = v["h2"]
    print(f"{key}:")
    print(f"  H1 GO (matched, >=2/3 scores): {h1go}")
    print(f"  H2 acc-top={h2['top_by_accuracy']} burden-top={h2['top_by_burden']} | "
          f"inversion_point={h2['inversion_point']} inversion_REAL={h2['inversion_real']} "
          f"(Δcov_gap={h2['inversion_diff']:+.4f} CI{h2['inversion_diff_ci']})")
    for m, mm in v["h3"]["methods"].items():
        labels = {sc: r["failure_type"] for sc, r in mm["per_score"].items()}
        print(f"  H3 {m}: divergence {labels} | set-size inflates under shift={mm['setsize_inflates_under_shift']}")

## 7. Show RESULTS_study.md + figures

In [ ]:
from IPython.display import Image, Markdown, display
display(Markdown(open("RESULTS_study.md", encoding="utf-8").read()))
for p in figs: display(Image(p))

## 8. STOP — cheap-first arms complete
Last-layer arms on Waterbirds + CelebA done and reported. **STOP for review.** Do NOT run the heavy
full-GroupDRO ResNet-50 fine-tune or any 3rd/4th dataset until sign-off. Excluded/flagged arms and
CelebA skips are recorded above and written to BLOCKERS.md — report honestly, do not re-tune.

In [ ]:
SKIPPED = sorted({d for _, d, _ in skipped})
if out["excluded"] or out["flagged"] or SKIPPED:
    lines = ["# BLOCKERS.md — grid arm exclusions / flags / skips\n"]
    for e in out["excluded"]:
        lines.append(f"- EXCLUDED {e['backbone']}/{e['dataset']} {e['method']} seed{e['seed']}: "
                     f"worst-group acc {e['worst_group_acc']:.3f} < floor {e['floor']} ({e['reason']})")
    for fl in out["flagged"]:
        lines.append(f"- FLAGGED {fl['backbone']}/{fl['dataset']} {fl['method']} seed{fl['seed']}: "
                     f"worst-group acc {fl['worst_group_acc']:.3f} < expected {fl['expected_min']} (kept for review)")
    for d in SKIPPED:
        lines.append(f"- SKIPPED dataset {d}: not available in this run.")
    open("BLOCKERS.md", "w", encoding="utf-8").write("\n".join(lines) + "\n")
    print("wrote BLOCKERS.md")
print("\nCHEAP-FIRST GRID COMPLETE. STOP for human review. Do NOT run heavy full-GroupDRO / 3rd-4th dataset.")